# 01. Train experiments

이 노트북은 학습만 담당합니다. 아래 설정을 바꾼 뒤 실행하면 결과와 checkpoint가 experiment/seed별 파일로 저장됩니다. 그래프와 표는 `02_analyze_experiments.ipynb`에서 생성합니다.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (Path('D:/gt-super') / 'data').exists():
    ROOT = Path('D:/gt-super')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Python 파일을 수정한 뒤에도 커널의 이전 클래스가 남지 않도록 의존성 순서대로 reload합니다.
import importlib
import gt_aux.config as config_module
import gt_aux.data as data_module
import gt_aux.model as model_module
import gt_aux.eval as eval_module
import gt_aux.train as train_module

config_module = importlib.reload(config_module)
data_module = importlib.reload(data_module)
model_module = importlib.reload(model_module)
eval_module = importlib.reload(eval_module)
train_module = importlib.reload(train_module)

ExperimentConfig = config_module.ExperimentConfig
prepare_data = data_module.prepare_data
release_model = train_module.release_model
train_one_experiment = train_module.train_one_experiment

d:\gt-super\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 학습 설정

여러 seed 실험은 `SEED`만 바꾸어 노트북을 다시 실행합니다. 같은 seed 집합을 모든 experiment에 사용해야 공정하게 비교할 수 있습니다.

In [2]:
RUN_MODE = 'full'
SEED = 42
DATA_SEED = 42  # 모든 training seed에서 고정: 동일 train/val split 보장
EXPERIMENTS = ['baseline', 'shared_detach', 'shared_e2e']

TRAIN_IMAGES = 3000
VAL_IMAGES = 750
EPOCHS = 20
BATCH_SIZE = 2
NUM_WORKERS = 0

IMAGE_MIN_SIZE = 640
IMAGE_MAX_SIZE = 1000
LEARNING_RATE = 2e-4
BACKBONE_LEARNING_RATE = 2e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 0.1
AUX_WEIGHT = 0.5
FEATURE_LEVEL = 0
HORIZONTAL_FLIP_P = 0.5
USE_AMP = None  # CUDA에서는 자동 활성화, CPU에서는 자동 비활성화
DETERMINISTIC = True
SAVE_EPOCH_CHECKPOINTS = False  # True면 epoch별 checkpoint가 추가로 쌓임
RESUME_FROM = {}  # 예: {'shared_e2e': ROOT / 'cache/checkpoints/shared_e2e_full/seed_42/checkpoint_full_shared_e2e_seed42.pt'}

CONFIG = ExperimentConfig.for_run(
    ROOT, run_mode=RUN_MODE, seed=SEED, data_seed=DATA_SEED, experiments=EXPERIMENTS,
    train_images=TRAIN_IMAGES, val_images=VAL_IMAGES, epochs=EPOCHS,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    image_min_size=IMAGE_MIN_SIZE, image_max_size=IMAGE_MAX_SIZE,
    lr=LEARNING_RATE, backbone_lr=BACKBONE_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
    base_aux_weight=AUX_WEIGHT, feature_level=FEATURE_LEVEL,
    horizontal_flip_p=HORIZONTAL_FLIP_P, use_amp=USE_AMP,
    deterministic=DETERMINISTIC,
    save_epoch_checkpoints=SAVE_EPOCH_CHECKPOINTS,
)
CONFIG.as_dict()

{'root': 'D:\\gt-super',
 'run_mode': 'full',
 'checkpoint': 'SenseTime/deformable-detr',
 'train_images': 3000,
 'val_images': 750,
 'epochs': 20,
 'batch_size': 2,
 'num_workers': 0,
 'image_size': {'shortest_edge': 640, 'longest_edge': 1000},
 'lr': 0.0002,
 'backbone_lr': 2e-05,
 'weight_decay': 0.0001,
 'grad_clip': 0.1,
 'base_aux_weight': 0.5,
 'feature_level': 0,
 'horizontal_flip_p': 0.5,
 'use_amp': True,
 'deterministic': True,
 'save_epoch_checkpoints': False,
 'checkpoint_group': None,
 'device': 'cuda',
 'experiments': ['baseline', 'shared_detach', 'shared_e2e'],
 'seed': 42,
 'data_seed': 42}

In [3]:
BUNDLE = prepare_data(CONFIG)
print({'train_images': len(BUNDLE.train_records), 'val_images': len(BUNDLE.val_records)})

VOC XML: 100%|██████████| 3750/3750 [00:08<00:00, 456.52it/s]


Full split: train=3000 (9180 objects), val=750 (2530 objects)
Current run: train=3000, val=750
{'train_images': 3000, 'val_images': 750}


## 1. Baseline 학습


In [4]:
baseline_model, baseline_history, baseline_gradients = train_one_experiment(
    CONFIG, BUNDLE, experiment='baseline', seed=CONFIG.seed,
    resume_from=RESUME_FROM.get('baseline'),
)
baseline_final_map = float(baseline_history.iloc[-1]['map'])
baseline_model = release_model(baseline_model)
print({'experiment': 'baseline', 'seed': CONFIG.seed, 'final_mAP': baseline_final_map,
       'checkpoint': str(CONFIG.checkpoint_path('baseline'))})


===== baseline / seed=42 =====


Loading weights: 100%|██████████| 545/545 [00:00<00:00, 10092.12it/s]
[transformers] DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
mod

[phase] initial main-only validation: 375 batches


[phase] initial validation complete: mAP=0.0001, AP@0.5=0.0002
[phase] training epoch 1/20: 1500 batches


baseline e1:   0%|          | 0/1500 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\transformers\cuda\attention_backwar

KeyboardInterrupt: 

## 2. Shared-Detach 학습


In [5]:
detach_model, detach_history, detach_gradients = train_one_experiment(
    CONFIG, BUNDLE, experiment='shared_detach', seed=CONFIG.seed,
    resume_from=RESUME_FROM.get('shared_detach'),
)
detach_final_map = float(detach_history.iloc[-1]['map'])
detach_model = release_model(detach_model)
print({'experiment': 'shared_detach', 'seed': CONFIG.seed, 'final_mAP': detach_final_map,
       'checkpoint': str(CONFIG.checkpoint_path('shared_detach'))})


===== shared_detach / seed=42 =====


Loading weights: 100%|██████████| 545/545 [00:00<00:00, 15525.39it/s]
[transformers] DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
mod

[phase] initial main-only validation: 50 batches


[phase] initial validation complete: mAP=0.0003, AP@0.5=0.0005
[phase] training epoch 1/7: 200 batches


shared_detach e1:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


...

[phase] validating epoch 6/7: 50 batches


{'epoch': 6, 'main_loss': 1.5573, 'aux_loss': 1.2868, 'aux_coverage': 0.9985, 'collision_rate': 0.0015, 'map': 0.0785, 'map50': 0.1312, 'map75': 0.0864}
[phase] training epoch 7/7: 200 batches


shared_detach e7:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[phase] validating epoch 7/7: 50 batches


{'epoch': 7, 'main_loss': 1.518, 'aux_loss': 1.1986, 'aux_coverage': 0.9992, 'collision_rate': 0.0008, 'map': 0.0805, 'map50': 0.1387, 'map75': 0.0862}
{'experiment': 'shared_detach', 'seed': 42, 'final_mAP': 0.08054248243570328, 'checkpoint': 'D:\\gt-super\\cache\\checkpoints\\checkpoint_smoke_shared_detach_seed42.pt'}


## 3. Shared-E2E 학습


In [6]:
e2e_model, e2e_history, e2e_gradients = train_one_experiment(
    CONFIG, BUNDLE, experiment='shared_e2e', seed=CONFIG.seed,
    resume_from=RESUME_FROM.get('shared_e2e'),
)
e2e_final_map = float(e2e_history.iloc[-1]['map'])
e2e_model = release_model(e2e_model)
print({'experiment': 'shared_e2e', 'seed': CONFIG.seed, 'final_mAP': e2e_final_map,
       'checkpoint': str(CONFIG.checkpoint_path('shared_e2e'))})


===== shared_e2e / seed=42 =====


Loading weights: 100%|██████████| 545/545 [00:00<00:00, 17768.61it/s]
[transformers] DeformableDetrForObjectDetection LOAD REPORT from: SenseTime/deformable-detr
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                         
mod

[phase] initial main-only validation: 50 batches


[phase] initial validation complete: mAP=0.0003, AP@0.5=0.0005
[phase] training epoch 1/7: 200 batches


shared_e2e e1:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


...

[phase] validating epoch 6/7: 50 batches


{'epoch': 6, 'main_loss': 1.4051, 'aux_loss': 0.9213, 'aux_coverage': 0.9985, 'collision_rate': 0.0015, 'map': 0.072, 'map50': 0.115, 'map75': 0.0783}
[phase] training epoch 7/7: 200 batches


shared_e2e e7:   0%|          | 0/200 [00:00<?, ?it/s]d:\gt-super\.venv\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[phase] validating epoch 7/7: 50 batches


{'epoch': 7, 'main_loss': 1.3539, 'aux_loss': 0.8557, 'aux_coverage': 0.9992, 'collision_rate': 0.0008, 'map': 0.0696, 'map50': 0.1146, 'map75': 0.0754}
{'experiment': 'shared_e2e', 'seed': 42, 'final_mAP': 0.06964635848999023, 'checkpoint': 'D:\\gt-super\\cache\\checkpoints\\checkpoint_smoke_shared_e2e_seed42.pt'}
